# Build your own index

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/05_build_your_own_index.ipynb)

An index in this package is **one record**: a function that turns a satellite image into a
single band, plus how that band should be shown. Register it and everything downstream —
`animate()`, the CLI, the Gradio GUI, the frame header, the colorbar — picks it up with no
further edits. `products.get_product` looks the name up **when you call it**, so an index
registered from a notebook cell is as real as one shipped in the package.

We build two, because they fail differently:

| | index | what it teaches |
|---|---|---|
| **Case 1** | **SAVI** — soil-adjusted vegetation index | the normal path: six harmonised band aliases, a tunable parameter, a viz range you have to justify |
| **Case 2** | **CIre** — red-edge chlorophyll index | the band the alias table *cannot* carry: raw band select, manual scaling, and a second registry 250 lines away |

Nothing here edits the package — all of it lives in this notebook.

## 1 · Setup and authentication

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"
    !pip install -q --force-reinstall --no-deps "{URL}"

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"
MONTH = ("2025-07-01", "2025-08-01")   # end exclusive -> exactly one frame

In [ ]:
from pathlib import Path

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / "aoi").is_dir()), None)

from gee_animation import auth
auth.init(PROJECT)

if REPO is not None:
    REGION = str(REPO / "docs" / "aoi" / "r12" / "r12_footprint.geojson")
else:
    import urllib.request
    RAW = (f"https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/{BRANCH}"
           "/docs/aoi/r12/r12_footprint.geojson")
    urllib.request.urlretrieve(RAW, "r12_footprint.geojson")
    REGION = "r12_footprint.geojson"
print("region:", REGION)

## 2 · Anatomy of an `Index`

Rather than describe the record, print a real one. Everything that makes NDVI work is here,
and your index needs the same fields filled in with the same care.

In [ ]:
from dataclasses import fields
from gee_animation import products

ndvi = products.INDICES["ndvi"]
for f in fields(ndvi):
    v = getattr(ndvi, f.name)
    if v in ((), "", None, False):
        continue                     # skip the optional hooks NDVI doesn't use
    print(f"{f.name:20s} {str(v)[:95]}")

print()
print("index band name:", products.INDEX_BAND)
print("registered now: ", sorted(products.INDICES))

Three of those fields are not decoration:

- **`compute`** — `(sensor, image, ee_module) -> ee.Image` with a single band named
  `products.INDEX_BAND`. Every later stage (`compositing`, `charts`, `metadata`, `render`)
  selects that band by name, so renaming is not optional.
- **`sensors`** — a frozenset, and the **only** gate on which satellite may be asked for this
  index. No other file carries a case for it.
- **`default_viz`** — `(min, max, palette)`. This is a *claim about the data*, applied
  identically to every frame so that colour is comparable across a run. NDVI's range came
  from a numeric sweep with pinned criteria (water separable from no-data grey,
  deuteranopia-safe endpoints). A range you invented is a guess — say so, and check it
  against real imagery before anyone cites the picture.

## 3 · Case 1 — SAVI

$$\text{SAVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red} + L}\,(1 + L)$$

NDVI saturates over dense canopy and exaggerates differences where soil shows through;
SAVI damps the soil background with $L$ (0.5 for mixed cover, 0 reduces it to NDVI exactly).

`sensor.reflectance(image, ee_module)` hands back exactly **six harmonised aliases** —
`blue green red nir swir1 swir2` — already scaled per sensor (Sentinel-2 ×0.0001,
Landsat C2 L2 ×0.0000275 − 0.2). Staying inside those six is what makes an index work on
more than one satellite.

In [ ]:
import ee
from gee_animation.products import INDEX_BAND

SAVI_L = 0.5     # soil brightness factor: 0 == NDVI, 1 for very sparse cover

def savi(sensor, image, ee_module=ee):
    """(NIR - Red) / (NIR + Red + L) * (1 + L), on harmonised reflectance."""
    refl = sensor.reflectance(image, ee_module)
    out = refl.expression(
        "((nir - red) / (nir + red + L)) * (1 + L)",
        {"nir": refl.select("nir"), "red": refl.select("red"), "L": SAVI_L},
    ).rename(INDEX_BAND)                                   # the band name the pipeline selects
    return out.set("system:time_start", image.get("system:time_start"))   # keep the clock

The last line is the one people forget. `compositing` groups scenes into months by
`system:time_start`; an index that drops it produces images the pipeline cannot place in
time, and the run fails far away from the cause.

## 4 · Register it

Insert into `products.INDICES` — the same dict the package fills at import. From here on
`"savi"` is a first-class index name everywhere in the library.

In [ ]:
from gee_animation.products import Index

products.INDICES["savi"] = Index(
    "savi",
    frozenset({"sentinel2", "landsat", "modis"}),   # the only availability gate
    # viz range: a GUESS — NDVI's palette, reused. Section 6 asks the data whether it
    # fits. Never let a copied ramp ship looking as measured as the one it came from.
    (-0.2, 1.0, ["#4575b4", "#aeaec7", "#8c510a", "#d8b365",
                 "#f6e8c3", "#41ab5d", "#006d2c"]),
    savi,
    bands="NIR, Red",
    formula=f"((NIR - Red) / (NIR + Red + L)) * (1 + L), L={SAVI_L}",
    display_name="Soil-adjusted vegetation (SAVI)",
    low_label="bare soil",
    high_label="dense vegetation",
)

from gee_animation.products import get_product, native_scale_m
print(get_product("sentinel2", "savi")[1].display_name)
print("native scale:", native_scale_m("sentinel2", "savi"), "m")
try:
    get_product("dynamicworld", "savi")          # outside the frozenset
except ValueError as e:
    print("rejected as expected:", e)

## 5 · Render it

One call, no other change. The header text and the colorbar's word anchors come straight out of the record registered above.

In [ ]:
from gee_animation import animate
from IPython.display import Image, display

view = animate(
    REGION, sensor="sentinel2", index="savi",
    start=MONTH[0], end=MONTH[1], project=PROJECT,
    buffer_m=1000.0, preset="1080p", fps=1, chart=False,
    out_dir="out/custom_index", name="savi_demo",
)
print(view.status)
display(Image(filename=sorted(p for p in view.frames if "_raw_" not in p)[0]))

## 6 · Check the ramp you guessed

The palette above was copied from NDVI, which is exactly the move that ships a wrong claim.
Ask the data instead: reduce both indices over the AOI for the same month and compare.
One Earth Engine round-trip, no rendering.

In [ ]:
from gee_animation import aoi, collection, compositing
from gee_animation.config import RunConfig

REGION_AOI = {"geojson": REGION} if isinstance(REGION, str) else {"bbox": REGION}
region = aoi.parse(REGION_AOI)

def aoi_stats(index_name):
    # composite one month through the real pipeline, then reduce it over the AOI
    frame_bbox = list(aoi.frame_bbox_from_region(region, 1000.0))
    frame = aoi.parse({"bbox": frame_bbox})
    cfg = RunConfig(
        name="stats", project=PROJECT,
        frame_aoi={"bbox": frame_bbox}, region_aoi=REGION_AOI,
        start=MONTH[0], end=MONTH[1], sensor="sentinel2", index=index_name,
        cadence="monthly", max_cloud_percent=80.0, region_max_cloud_percent=60.0,
        viz_min=0.0, viz_max=1.0, palette=[], fps=1.0, scale=10, dimensions=768)
    month = compositing.monthly_median(collection.build(cfg, frame, region), cfg)[0]
    return month.image.select(INDEX_BAND).reduceRegion(
        reducer=ee.Reducer.percentile([5, 50, 95]), geometry=region, scale=20,
        maxPixels=int(1e8), bestEffort=True).getInfo()


for name in ("ndvi", "savi"):
    s = aoi_stats(name)
    lo, mid, hi = (s[f"{INDEX_BAND}_p{p}"] for p in (5, 50, 95))
    print(f"{name.upper():5s}  p5 {lo:+.3f}   median {mid:+.3f}   p95 {hi:+.3f}")

Measured over this AOI in July 2025:

```
NDVI   p5 +0.744   median +0.854   p95 +0.904
SAVI   p5 +0.409   median +0.529   p95 +0.627
```

Same forest, same month, and **SAVI lands a third of a ramp lower than NDVI** — exactly what
`(1+L)/(NIR+Red+L)` does to a high-NDVI canopy. On the borrowed palette those values fall in
the browns and pale yellows, whose word anchor says *bare soil*. The picture in section 5 is
therefore mildly dishonest: it renders closed forest in the colours NDVI reserves for
clear-cut.

Two defensible fixes, and the choice is a decision you write down rather than a default:

- **measure it** — give SAVI its own range (roughly `0.3 … 0.7` over this AOI) so the palette
  spends its colours where the data actually lives; or
- **keep NDVI's range on purpose**, because you want the two products directly comparable, and
  accept that SAVI always reads cooler.

Either way the ramp stops being a guess. Note too that both indices crowd the top of
`-0.2 … 1.0`: over temperate forest in midsummer, half of NDVI's ramp is never used.

## 7 · Case 2 — the band the alias table cannot carry

$$\text{CIre} = \frac{\text{NIR}}{\text{RedEdge}} - 1$$

The red edge (Sentinel-2 **B5**, 705 nm) tracks chlorophyll and keeps responding where NDVI
has saturated. It is also **not one of the six aliases**, because `reflectance()` only carries
bands every reflectance sensor has — and neither Landsat nor MODIS has a red edge.

So three things change at once:

1. B5 is selected straight off the raw image and **scaled by hand** (Sentinel-2: ×0.0001 — get
   this wrong and the ratio is off by four orders of magnitude);
2. `sensors` shrinks to Sentinel-2 alone;
3. B5 is a **20 m** band, and the GSD table is a *second registry*, `_S2_20M_INDICES`.

This follows `products._ndre`, the existing precedent for a non-aliased band.

In [ ]:
def cire(sensor, image, ee_module=ee):
    """NIR / RedEdge - 1, Sentinel-2 only: B5 is outside the shared alias table."""
    refl = sensor.reflectance(image, ee_module)
    red_edge = image.select("B5").multiply(0.0001).rename("red_edge")   # scale by hand
    combo = refl.select("nir").rename("nir").addBands(red_edge)
    out = combo.expression(
        "nir / red_edge - 1",
        {"nir": combo.select("nir"), "red_edge": combo.select("red_edge")},
    ).rename(INDEX_BAND)
    return out.set("system:time_start", image.get("system:time_start"))


products.INDICES["cire"] = Index(
    "cire",
    frozenset({"sentinel2"}),            # red edge exists on no other sensor here
    # A RATIO index: unbounded below ~0 and up past 8 over dense canopy — a symmetric
    # -1..1 range would be meaningless. Range below is a guess for temperate forest.
    (0.0, 6.0, ["#f7f4e9", "#d8b365", "#41ab5d", "#006d2c"]),
    cire,
    bands="NIR (B8), Red Edge (B5)",
    formula="B8 / B5 - 1",
    display_name="Red-edge chlorophyll index (CIre)",
    low_label="bare / stressed",
    high_label="chlorophyll-rich",
)
print(get_product("sentinel2", "cire")[1].display_name)

### The registry 250 lines away

`native_scale_m` is what stops a render from pretending to more detail than the sensor has.
It reads `_S2_20M_INDICES`, which is a **frozenset** — you rebind it, you cannot mutate it.
Skip this and CIre renders on a 10 m grid, i.e. Earth Engine interpolating a 20 m band and
every pixel implying a precision the measurement does not have.

In [ ]:
print("before:", native_scale_m("sentinel2", "cire"), "m   (wrong: B5 is a 20 m band)")

products._S2_20M_INDICES = products._S2_20M_INDICES | {"cire"}   # rebind, not mutate

print("after: ", native_scale_m("sentinel2", "cire"), "m")

In [ ]:
view_cire = animate(
    REGION, sensor="sentinel2", index="cire",
    start=MONTH[0], end=MONTH[1], project=PROJECT,
    buffer_m=1000.0, preset="1080p", fps=1, chart=False,
    out_dir="out/custom_index", name="cire_demo",
)
print(view_cire.status)
display(Image(filename=sorted(p for p in view_cire.frames if "_raw_" not in p)[0]))

Note what the palette says about the lake: a **ratio** index over water is near zero, so
`low_label="bare / stressed"` labels open water as stressed vegetation. Normalized-difference
indices separate water naturally (NDVI drives it negative, which is why NDVI's palette has a
blue end); a ratio index does not. If this product mattered you would mask water, or say so
in the legend — a word anchor is a claim too.

## 8 · What to carry away

| Gotcha | Symptom if ignored |
|---|---|
| band must be renamed to `products.INDEX_BAND` | later stages select a band that isn't there |
| carry `system:time_start` | frames cannot be placed in a month; failure appears far from the cause |
| only six aliases exist (`blue green red nir swir1 swir2`) | any other band must be selected raw **and scaled per sensor** |
| `sensors` frozenset is the only availability gate | an index silently offered for a satellite that lacks the band |
| `default_viz` is a claim, not a style | a guessed ramp ships looking as authoritative as a measured one |
| GSD lives in a second registry (`_S2_20M_INDICES`, `THERMAL_INDICES`) | a 20 m or 100 m band rendered as if it were 10 m |

Everything above lived in this notebook, which is the point: nothing in the package needed
changing for a new index to render, chart and export like a built-in one. The same record,
written into `gee_animation/products.py` instead of a cell, is how `ndvi`, `nbr` and `lst`
are defined — open that file and the entries will now read like something you have written.

## 9 · Make it yours

1. **Turn the knob.** SAVI's whole point is its soil factor `L` — set it to
   `0.25` and `1.0`, re-render §5's frame, and watch the ramp shift where the
   canopy is thin.
2. **A different formula.** Register GNDVI — `(nir − green)/(nir + green)` —
   under its own name and render it next to SAVI for the same month. Two
   registrations, one pipeline, zero downstream edits.
3. **Interrogate your ramp.** Re-run §6's measurement for `MONTH =
   ("2025-01-01", "2025-02-01")`. Does the range you picked in July survive
   January? A viz range is a seasonal claim, not a constant.

*The seam behind it:* the registry in `gee_animation/products.py` — `INDICES`,
read by `get_product` at call time. Every notebook in this series, the CLI, the
GUI and the colorbar all derive from that one dict; what you registered here is
already everywhere.

## 10 · Graduate to a pull request

Its short form:

- the compute function + `INDICES` entry **with its `sensors` frozenset** — and
  the two registries 250 lines away (`_S2_20M_INDICES`, `THERMAL_INDICES`) if
  your band is 20 m or thermal;
- `display_name`, `low_label`/`high_label`, `units`, `bands`, `formula` — a
  product that renders correctly but reads badly is not done;
- a **measured** `default_viz` (§6 of this notebook is exactly that
  measurement), or one flagged as a guess;
- `tests/test_products.py::test_registry_contents` updated **in the same
  commit** — it pins the registry by name and fails otherwise — plus a maths
  test and a sensor-restriction test of your own, and a green
  `pytest -m "not integration"`;
- `config/<name>.example.yaml` and a row in the README's index table.

A pull request that follows that list is reviewable in minutes — every box is a
thing the reviewer no longer has to discover.